#**1st Week**

##**Задачи - Оконные функции (Продвинутый уровень)**

В этом тесте вам предстоит решить практические задачи продвинутого уровня на тему "Оконные функции".

Для всех комнат вывести номер комнаты, ее тип, вместимость и колонку avg_occupancy, в которой будет указана средняя вместимость по данному типу комнат. Отсортировать по убыванию средней вместимости данного типа комнат

In [ ]:
SELECT room_number, 
       type_name, 
       max_occupancy, 
       AVG(max_occupancy) OVER (PARTITION BY type_name) AS avg_occupancy
FROM rooms
ORDER BY avg_occupancy DESC;

Вывести для каждой комнаты её номер, площадь, вместимость и колонку avg_size_for_occupancy, в которой будет указана средняя площадь комнат с такой же вместимостью

In [ ]:
SELECT room_number, 
       room_size_sqm, 
       max_occupancy, 
       AVG(room_size_sqm) OVER (PARTITION BY max_occupancy) AS avg_size_for_occupancy
FROM rooms;

Вывести сумму всех платежей за каждый день в 2017 году . Отобразить id платежа, дату оплаты и колонку total_payments_per_day, в которой будет указана сумма всех оплат за день рассматриваемого платежа.

In [ ]:
SELECT payment_id, 
       payment_date, 
       SUM(amount_paid) OVER (PARTITION BY payment_date) AS total_payments_per_day
FROM payments
WHERE payment_date >= '2017-01-01' AND payment_date < '2018-01-01';

Для каждого бронирования добавить колонку, в которой будет показано количество бронирований этим клиентом. В результате должны получиться 3 колонки: booking_id, renter_id и bookings_count (то самое количество бронирований). Отобразить данные только для первых 60 клиентов (renter_id 1-60). Отсортировать по booking_id.

In [ ]:
SELECT b.booking_id, 
       b.renter_id, 
       COUNT(b2.booking_id) AS bookings_count
FROM bookings b
JOIN bookings b2 ON b.renter_id = b2.renter_id
WHERE b.renter_id BETWEEN 1 AND 60
GROUP BY b.booking_id, b.renter_id
ORDER BY b.booking_id;

Для каждой комнаты вывести номер комнаты, ее тип, площадь и колонку avg_room_size_sqm, в которой будет указана средняя площадь комнаты данного типа

In [ ]:
SELECT room_number, 
       type_name, 
       room_size_sqm, 
       AVG(room_size_sqm) OVER (PARTITION BY type_name) AS avg_room_size_sqm
FROM rooms;

Для каждой комнаты определить, на сколько процентов ее площадь больше, чем средняя площадь комнат того же типа. Нормировать нужно именно на среднюю площадь. Вывести номер комнаты, ее тип, площадь и колонку size_difference_percentage, в которой будет указана процентная разность площадей. Колонку с процентом округлить до двух знаков с помощью встроенной функции ROUND. Для комнат, меньших среднего, значение в этой колонке будет отрицательным.

Примечание: в SQL при делении целого числа на целое получается целое число, т.е. вся дробная часть округляется. Чтобы этого избежать, можно либо привести значение к типу REAL (погуглить про оператор CAST), либо первым множителем выражения сделать дробное число. Раз уже речь идет о процентах, то можно использовать число 100.0. 

In [ ]:
SELECT room_number, 
       type_name, 
       room_size_sqm, 
       ROUND(((room_size_sqm - avg_room_size) / avg_room_size) * 100.0, 2) AS size_difference_percentage
FROM (
    SELECT room_number, 
           type_name, 
           room_size_sqm, 
           AVG(room_size_sqm) OVER (PARTITION BY type_name) AS avg_room_size
    FROM rooms
) AS subquery;

Для каждого платежа вывести номер платежа, способ оплаты, сумму оплаты и колонку avg_amount_paid, в которой будет указана средняя сумма оплаты по данному способу оплаты. Все подсчеты производить только по платежам с суммой оплаты более 200000.

In [ ]:
SELECT payment_id, 
       payment_method, 
       amount_paid, 
       AVG(amount_paid) OVER (PARTITION BY payment_method) AS avg_amount_paid
FROM payments
WHERE amount_paid > 200000;

Пронумеровать комнаты в порядке возрастания цены внутри каждого типа комнаты. Вывести номер комнаты, ее тип, цену и колонку с названием row_number, в которой будет указан порядковый номер, описанный условием ранее.

Примечание: дополнительно сортировать выборку не надо!

In [ ]:
SELECT room_number, 
       type_name, 
       price_per_night  , 
       ROW_NUMBER() OVER (PARTITION BY type_name ORDER BY price_per_night  ) AS row_number
FROM rooms;

Пронумеровать платежи в порядке убывания суммы платежа внутри каждого метода оплаты. Вывести номер платежа, сумму оплаты, метод оплаты и колонку с названием row_number, в которой будет указан порядковый номер, описанный условием ранее. Отобразить данные только для платежей с суммой оплаты более 200000

Примечание: дополнительно сортировать выборку не надо!

In [ ]:
SELECT payment_id, 
       amount_paid, 
       payment_method, 
       ROW_NUMBER() OVER (PARTITION BY payment_method ORDER BY amount_paid DESC) AS row_number
FROM payments
WHERE amount_paid > 200000;

Для каждого бронирования вывести его номер, дату заезда, дату выезда и колонку stay_time, в которой будет указана продолжительность пребывания в днях. Отобразить только бронирования с продолжительностью пребывания более 20 дней.

В postgres для работы с датами и таймстемпами можно использовать функцию date_trunc. Функцию Extract использовать не всегда стоит, потому что при выделении месяца пропадает год, и, например, в декабрьско-январских бронированиях могут возникнуть ошибки. Но ЛМС у нас использует sqlite, в которой нет подобных функций, поэтому вот подсказка:

Для решения данной задачи воспользуйтесь функцией JULIANDAY.

Пример: SELECT JULIANDAY(check_out_date); - для получения дня из даты выезда.

In [ ]:
SELECT booking_id, 
       check_in_date, 
       check_out_date, 
       JULIANDAY(check_out_date) - JULIANDAY(check_in_date) AS stay_time
FROM bookings
WHERE JULIANDAY(check_out_date) - JULIANDAY(check_in_date) > 20;

Найти тип комнаты с максимальным разбросом цен. Вывести тип комнаты и колонку с названием price_range, в которой будет указана разница между максимальной и минимальной ценой данного типа комнаты

In [ ]:
SELECT type_name, 
       MAX(price_per_night  ) - MIN(price_per_night  ) AS price_range
FROM rooms
GROUP BY type_name
ORDER BY price_range DESC
LIMIT 1;

Определите процент комнат каждого типа от общего количества комнат. Выведите все возможные типы комнат и колонку с названием percentage, в которой будет указан процент данного типа от общего количества комнат. Округлите результат до двух знаков после запятой, используя функцию ROUND.

Важно: в SQL, когда мы делим целое число на целое число, в результате получается целое число. Таким образом, дробная часть числа пропадает. Чтобы этого избежать, нужно, чтобы первый операнд в операции деления был типом REAL. Для этого его нужно либо привести к этому типу (гуглите CAST), либо, поскольку у нас речь идет о процентах, первым множителем сделать 100.0.

In [ ]:
SELECT type_name, 
       ROUND(CAST(COUNT(*) AS REAL) / (SELECT COUNT(*) FROM rooms) * 100.0, 2) AS percentage
FROM rooms
GROUP BY type_name;